# OmniParser v2 — Demo

VSCode 및 Google Colab 공용 노트북입니다.

- **VSCode**: Python 인터프리터를 가상환경으로 설정 후 실행
- **Colab**: 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 0. 환경 확인

In [ ]:
import subprocess, os, sys

# GPU 확인
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  GPU를 찾을 수 없습니다. CPU로 실행됩니다 (속도 저하 예상).')
else:
    print(result.stdout)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'   CUDA : {torch.version.cuda}')
else:
    print('ℹ️  CUDA 없음 — CPU 모드로 계속합니다.')

# 레포 루트를 작업 디렉토리로 설정
notebook_dir = os.path.dirname(os.path.abspath('colab_demo.ipynb'))
os.chdir(notebook_dir)
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)
print(f'작업 디렉토리: {os.getcwd()}')

## 1. 패키지 설치

> **최초 1회만 실행** — 이후 실행 시 이 셀은 건너뛰어도 됩니다.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip(
    'torch', 'torchvision',
    'easyocr',
    'transformers',
    'ultralytics==8.3.70',
    'supervision==0.18.0',
    'timm',
    'einops==0.8.0',
    'accelerate',
    'paddlepaddle',
    'paddleocr',
)
print('✅ 패키지 설치 완료')

## 2. 모델 가중치 다운로드

`weights/` 폴더가 이미 존재하면 건너뜁니다.

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('weights', exist_ok=True)

# OmniParser-v2.0 모델 가중치
model_files = [
    'icon_detect/train_args.yaml',
    'icon_detect/model.pt',
    'icon_detect/model.yaml',
    'icon_caption/config.json',
    'icon_caption/generation_config.json',
    'icon_caption/model.safetensors',
]
for f in model_files:
    if not os.path.exists(f'weights/{f}') and not os.path.exists('weights/icon_caption_florence/model.safetensors'):
        hf_hub_download(repo_id='microsoft/OmniParser-v2.0', filename=f, local_dir='weights')
        print(f'✓ {f}')

if os.path.exists('weights/icon_caption') and not os.path.exists('weights/icon_caption_florence'):
    shutil.move('weights/icon_caption', 'weights/icon_caption_florence')

# Florence-2 processor 파일 (로컬 로드용)
proc_files = [
    'tokenizer.json', 'tokenizer_config.json',
    'special_tokens_map.json', 'preprocessor_config.json',
]
for f in proc_files:
    dest = f'weights/icon_caption_florence/{f}'
    if not os.path.exists(dest):
        hf_hub_download(repo_id='microsoft/Florence-2-base', filename=f,
                        local_dir='weights/icon_caption_florence')
        print(f'✓ processor: {f}')

print('\n가중치 파일:')
for root, _, files in os.walk('weights'):
    if '.cache' in root:
        continue
    for file in sorted(files):
        print(f'  {os.path.join(root, file)}')

## 3. 모델 로드

In [ ]:
import torch
from PIL import Image
from util.utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')

som_model = get_yolo_model('weights/icon_detect/model.pt')
som_model.to(device)
print('✅ YOLO 모델 로드 완료')

caption_model_processor = get_caption_model_processor(
    model_name='florence2',
    model_name_or_path='weights/icon_caption_florence',
    device=device
)
print('✅ Florence2 모델 로드 완료')

## 4. 이미지 파싱

In [ ]:
import base64, io, time
import matplotlib.pyplot as plt

# 파싱할 이미지 경로 지정
image_path = 'imgs/windows_home.png'
# image_path = 'imgs/google_page.png'
# image_path = '/path/to/your/screenshot.png'

image = Image.open(image_path)
print(f'이미지 크기: {image.size}')

box_overlay_ratio = max(image.size) / 3200
draw_bbox_config = {
    'text_scale': 0.8 * box_overlay_ratio,
    'text_thickness': max(int(2 * box_overlay_ratio), 1),
    'text_padding': max(int(3 * box_overlay_ratio), 1),
    'thickness': max(int(3 * box_overlay_ratio), 1),
}

t0 = time.time()
ocr_bbox_rslt, _ = check_ocr_box(
    image_path, display_img=False, output_bb_format='xyxy',
    goal_filtering=None,
    easyocr_args={'paragraph': False, 'text_threshold': 0.9},
    use_paddleocr=True
)
text, ocr_bbox = ocr_bbox_rslt
print(f'OCR 완료: {time.time()-t0:.1f}s')

t1 = time.time()
labeled_img_b64, label_coords, parsed_content_list = get_som_labeled_img(
    image_path, som_model,
    BOX_TRESHOLD=0.05,
    output_coord_in_ratio=True,
    ocr_bbox=ocr_bbox,
    draw_bbox_config=draw_bbox_config,
    caption_model_processor=caption_model_processor,
    ocr_text=text,
    use_local_semantics=True,
    iou_threshold=0.7,
    scale_img=False,
    batch_size=128
)
print(f'파싱 완료: {time.time()-t1:.1f}s  |  감지 요소: {len(parsed_content_list)}개')

## 5. 결과 시각화

In [ ]:
import base64, io
import matplotlib.pyplot as plt
from PIL import Image

result_img = Image.open(io.BytesIO(base64.b64decode(labeled_img_b64)))
plt.figure(figsize=(16, 10))
plt.imshow(result_img)
plt.axis('off')
plt.title(f'OmniParser v2 — {len(parsed_content_list)}개 요소 감지', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(parsed_content_list)
df.index.name = 'ID'
df